# Nestlé Commodity Price Pipeline & Stock Analysis

**Goal:** Build an end-to-end data pipeline from the World Bank Pink Sheet, filter to commodities
relevant to Nestlé's cost base, enrich with statistical features, and analyse the relationship
between commodity price movements and Nestlé's stock price (NESN.SW).

**Data sources:**
- World Bank CMO Historical Monthly Data (Pink Sheet) — 71 commodities, Jan 1960 to Feb 2026
- Yahoo Finance — NESN.SW monthly adjusted close price

**Pipeline stages:**
1. Load and inspect raw Excel
2. Clean headers and parse dates
3. Filter to Nestlé-relevant commodities
4. Reshape wide to long, engineer features
5. Build star schema (dim + fact tables)
6. Merge with stock data and run correlation and regression analysis


---
## 1. Load Raw Data


In [1]:
import pandas as pd

FILE_PATH = '/Users/tsh/Downloads/CMO-Historical-Data-Monthly.xlsx'
raw = pd.read_excel(FILE_PATH, sheet_name='Monthly Prices', header=None)

print(f'Raw shape: {raw.shape}')
raw.iloc[:8, :6]  # preview: metadata rows + first few commodity columns


FileNotFoundError: [Errno 2] No such file or directory: '/Users/tsh/Downloads/CMO-Historical-Data-Monthly.xlsx'

---
## 2. Clean Headers and Parse Dates

The raw file has a non-standard layout:
- Rows 0-3: metadata (title, description, update date)
- Row 4: commodity names
- Row 5: units
- Row 6+: monthly price data

Steps: extract column names from row 4, slice data from row 6, convert World Bank date format
(`1960M01`) to proper `datetime`.


In [ ]:
# Extract commodity names and units from header rows
commodity_names = raw.iloc[4].tolist()
units_row = raw.iloc[5].tolist()

# Slice data rows only (row 6 onwards) and assign column headers
data = raw.iloc[6:].copy()
data.columns = commodity_names

# Rename the first column (currently NaN) to date
data = data.rename(columns={commodity_names[0]: "date"})

# Convert World Bank date format (1960M01) to proper datetime
data["date"] = pd.to_datetime(
    data["date"].astype(str).str.replace(r"(\d{4})M(\d{2})", r"\1-\2", regex=True),
    format="%Y-%m",
    errors="coerce"
)

# Drop rows where date parsing failed
data = data[data["date"].notna()].reset_index(drop=True)

print(f'Date range: {data["date"].min()} -> {data["date"].max()}')
print(f'Total rows: {len(data)}')


---
## 3. Filter to Nestlé-Relevant Commodities

From 71 available commodities, select the 8 that directly map to Nestlé's
key ingredient categories: Beverages, Confectionery, Sweeteners, Grains, Oils & Fats.


In [ ]:
Nestle_Commodities = {
    "Coffee, Arabica":  "Coffee Arabica",
    "Coffee, Robusta":  "Coffee Robusta",
    "Cocoa":            "Cocoa",
    "Sugar, world":     "Sugar (World)",
    "Wheat, US SRW":    "Wheat",
    "Maize":            "Corn (Maize)",
    "Palm oil":         "Palm Oil",
    "Soybeans":         "Soybeans",
}

cols_to_keep = ["date"] + list(Nestle_Commodities.keys())
df = data[cols_to_keep].copy()
df = df.rename(columns=Nestle_Commodities)

print(f'Shape after filter: {df.shape}')
df.head(3)


---
## 4. Reshape Wide to Long and Engineer Features

**Why long format?** Power BI and most BI tools work best with a single
`(date, commodity, price)` structure rather than one column per commodity.
`pd.melt()` pivots the 8 commodity columns into rows.

**Derived features added:**
- `year`, `month`, `month_name` — for time-based slicing
- `price_yoy_pct` — year-over-year % change (grouped per commodity to avoid bleed)
- `price_12m_avg` — 12-month rolling average (`min_periods=3`)


In [ ]:
# Wide to long pivot
df_long = df.melt(
    id_vars="date",
    var_name="commodity",
    value_name="price"
)

# Convert price to numeric (some early cells contain placeholder characters)
df_long["price"] = pd.to_numeric(df_long["price"], errors="coerce")
df_long = df_long.dropna(subset=["price"]).reset_index(drop=True)

# Sort and add date parts
df_long = df_long.sort_values(["commodity", "date"]).reset_index(drop=True)
df_long["year"]       = df_long["date"].dt.year
df_long["month"]      = df_long["date"].dt.month
df_long["month_name"] = df_long["date"].dt.strftime("%b")

# YoY % change grouped by commodity
df_long["price_yoy_pct"] = (
    df_long
    .groupby("commodity")["price"]
    .pct_change(periods=12)
    .mul(100)
    .round(2)
)

# 12-month rolling average
df_long["price_12m_avg"] = (
    df_long
    .groupby("commodity")["price"]
    .transform(lambda x: x.rolling(12, min_periods=3).mean())
    .round(4)
)

print(f'Long format shape: {df_long.shape}')
df_long.head(3)


---
## 5. Visualise and Export

Quick Plotly chart indexed to 100 at the start of the 15-year window — this makes
all commodities comparable on one Y axis regardless of unit ($/kg vs $/mt).
Then export the flat CSV and star schema tables.


In [ ]:
import plotly.express as px

# Last 15 years only — full 1960 history is too noisy for a trend chart
cutoff = df_long["date"].max() - pd.DateOffset(years=15)
df_plot = df_long[df_long["date"] >= cutoff].copy()

# Index to 100 at period start
def index_to_100(group):
    base = group["price"].iloc[0]
    group["price_indexed"] = (group["price"] / base * 100).round(2)
    return group

df_plot = df_plot.groupby("commodity", group_keys=False).apply(index_to_100)

fig = px.line(
    df_plot, x="date", y="price_indexed", color="commodity",
    title="Nestlé Key Commodity Prices — Last 15 Years (Indexed to 100)",
    labels={"price_indexed": "Price Index (base = 100)", "date": ""},
    template="plotly_white",
)
fig.add_hline(y=100, line_dash="dot", line_color="gray", opacity=0.4)
fig.show()


In [ ]:
# Flat CSV export
df_long.to_csv("nestle_commodity_prices.csv", index=False)
print(f"Exported {len(df_long):,} rows to nestle_commodity_prices.csv")

# Dimension table
category_map = {
    "Coffee Arabica": "Beverages",    "Coffee Robusta": "Beverages",
    "Cocoa":          "Confectionery","Sugar (World)":  "Sweeteners",
    "Wheat":          "Grains",       "Corn (Maize)":   "Grains",
    "Palm Oil":       "Oils & Fats",  "Soybeans":       "Oils & Fats",
}
unit_map = {
    "Coffee Arabica": "$/kg", "Coffee Robusta": "$/kg",
    "Cocoa":          "$/kg", "Sugar (World)":  "$/kg",
    "Wheat":          "$/mt", "Corn (Maize)":   "$/mt",
    "Palm Oil":       "$/mt", "Soybeans":       "$/mt",
}

dim_commodity = pd.DataFrame({"commodity_name": list(Nestle_Commodities.values())})
dim_commodity.insert(0, "commodity_key", range(1, len(dim_commodity) + 1))
dim_commodity["category"] = dim_commodity["commodity_name"].map(category_map)
dim_commodity["unit"]     = dim_commodity["commodity_name"].map(unit_map)

# Fact table
fact_prices = df_long.merge(
    dim_commodity[["commodity_key", "commodity_name"]],
    left_on="commodity", right_on="commodity_name", how="left"
)[
    ["date", "commodity_key", "price", "price_yoy_pct", "price_12m_avg"]
].rename(columns={"date": "date_key"})

dim_commodity.to_csv("dim_commodity.csv", index=False)
fact_prices.to_csv("fact_prices.csv", index=False)

print(f"dim_commodity : {dim_commodity.shape}")
print(f"fact_prices   : {fact_prices.shape}")
dim_commodity


---
## 6. Correlation and Regression: Commodities vs Nestlé Stock Price

**Hypothesis:** rising commodity input costs suppress Nestlé's margins and should
negatively correlate with its stock price.

**Method:**
- Download NESN.SW monthly adjusted prices via `yfinance`
- Merge with commodity data on `date` — inner join gives 427 months (Jan 1990 to Feb 2026)
- Convert all series to **monthly % returns** before analysis to remove long-run trend
- Run: correlation matrix, lagged correlation (0-6 months), OLS regression


In [ ]:
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns

# Download stock data
# auto_adjust=True pre-adjusts for dividends/splits and avoids MultiIndex columns
nestle = yf.download("NESN.SW", start="1960-01-01", end="2026-03-01",
                     interval="1mo", auto_adjust=True)
nestle = nestle[["Close"]].rename(columns={"Close": "stock_price"})

# Flatten any remaining MultiIndex and normalise dates to midnight
nestle.columns = [col[0] if isinstance(col, tuple) else col for col in nestle.columns]
nestle.index   = nestle.index.to_period("M").to_timestamp()
nestle         = nestle.reset_index().rename(columns={"Date": "date"})
nestle["date"] = pd.to_datetime(nestle["date"]).dt.normalize()

# Pivot commodities wide and normalise dates before merge
commodity_wide = df_long.pivot_table(
    index="date", columns="commodity", values="price"
).reset_index()
commodity_wide.columns.name = None
commodity_wide["date"] = pd.to_datetime(commodity_wide["date"]).dt.normalize()

df_merged = commodity_wide.merge(nestle, on="date", how="inner").dropna()

print(f'Merged shape: {df_merged.shape}')
print(f'Date range : {df_merged["date"].min()} -> {df_merged["date"].max()}')


In [ ]:
feature_cols = ["Cocoa", "Coffee Arabica", "Coffee Robusta", "Corn (Maize)",
                "Palm Oil", "Soybeans", "Sugar (World)", "Wheat"]

# Convert to monthly % returns
df_returns = df_merged[feature_cols + ["stock_price"]].pct_change().dropna()

# Correlation heatmap
corr_matrix = df_returns.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Monthly Returns: Nestlé Commodities vs Stock Price")
plt.tight_layout()
plt.show()

print("Correlation with stock_price:")
print(corr_matrix["stock_price"].drop("stock_price").sort_values())


In [ ]:
# Lagged correlation — tests whether commodity moves today predict stock returns later
results = []
for commodity in feature_cols:
    for lag in range(0, 7):
        corr = df_returns[commodity].shift(lag).corr(df_returns["stock_price"])
        results.append({"commodity": commodity, "lag_months": lag, "correlation": round(corr, 4)})

lag_df    = pd.DataFrame(results)
lag_pivot = lag_df.pivot(index="commodity", columns="lag_months", values="correlation")
lag_pivot.columns = [f"lag_{c}m" for c in lag_pivot.columns]

plt.figure(figsize=(10, 6))
sns.heatmap(lag_pivot, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Lagged Correlation: Commodity Returns vs Nestlé Stock Price")
plt.tight_layout()
plt.show()

print(lag_pivot.sort_values("lag_0m"))


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import statsmodels.api as sm

# Cocoa has the strongest immediate signal (lag_0)
# Coffee Arabica peaks at lag_1 — likely a demand signal rather than cost signal
df_reg = pd.DataFrame({
    "cocoa_lag0":  df_returns["Cocoa"].shift(0),
    "coffee_lag1": df_returns["Coffee Arabica"].shift(1),
    "stock_price": df_returns["stock_price"]
}).dropna()

X = df_reg[["cocoa_lag0", "coffee_lag1"]]
y = df_reg["stock_price"]

# sklearn: quick coefficient and R² check
sk_model = LinearRegression().fit(X, y)
print("=== Regression Results ===")
print(f"Cocoa (lag 0) coeff :  {sk_model.coef_[0]:.4f}")
print(f"Coffee (lag 1) coeff:  {sk_model.coef_[1]:.4f}")
print(f"Intercept            : {sk_model.intercept_:.4f}")
print(f"R2                   : {r2_score(y, sk_model.predict(X)):.4f}")

# statsmodels: p-values and confidence intervals
ols = sm.OLS(y, sm.add_constant(X)).fit()
print(ols.summary())

# Actual vs predicted scatter
plt.figure(figsize=(8, 5))
plt.scatter(sk_model.predict(X), y, alpha=0.4)
plt.axline((0, 0), slope=1, color="red", linestyle="--", label="perfect fit")
plt.xlabel("Predicted stock return")
plt.ylabel("Actual stock return")
plt.title("Nestlé Stock: Actual vs Predicted Monthly Return")
plt.legend()
plt.tight_layout()
plt.show()


---
## Key Findings

| Commodity | Best lag | Correlation with stock |
|-----------|----------|------------------------|
| Cocoa | lag_0m | **-0.16** |
| Coffee Arabica | lag_1m | +0.10 |
| All others | — | < ±0.05 |

**Regression R² = 0.038** — commodity prices explain ~4% of monthly stock return variance.
Typical for a large diversified FMCG company with hedging programs and 400+ brands.
Cocoa is the only input with a clear, immediate negative relationship with the stock.

**To improve the model:** add EUR/CHF FX rate, a broad market index (S&P 500 / MSCI World),
and Nestlé earnings release indicators as control variables.
